# Rider Counting — NEW pipeline (方法2: 逐帧检测 + 近邻帧去重)

新方法运行入口（notebook 版）。逻辑与 `scripts/run_rider_count_new.py` 完全同源（直接 import，不是复制）。

**用法**：改 Cell 2 的配置 → 依次运行。单地点跑 Cell 3–4；全量批跑用 Cell 5。

In [ ]:
# Cell 1: Bootstrap — 定位仓库根目录并加载新管线模块 (健壮版)
import sys
import importlib.util
from pathlib import Path

p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
assert (p / "src").exists(), "找不到包含 src/ 的仓库根目录 — 请把本 notebook 放在仓库的 notebooks/ 文件夹里"
REPO_ROOT = p
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

script = REPO_ROOT / "scripts" / "run_rider_count_new.py"
if not script.exists():
    hits = [h for h in REPO_ROOT.rglob("run_rider_count_new.py")]
    print("!! scripts/run_rider_count_new.py 不在预期位置")
    print("   搜索到:", hits if hits else "(整个仓库里都没有)")
    assert hits, "请把 zip 里的 scripts/ 文件夹解压到仓库根目录 (和 src/ 同级)"
    script = hits[0]
    print("   使用:", script)

spec = importlib.util.spec_from_file_location("run_rider_count_new", script)
rrc = importlib.util.module_from_spec(spec)
sys.modules["run_rider_count_new"] = rrc   # dataclass 需要模块已注册
spec.loader.exec_module(rrc)
print("模块加载成功:", script.name)

In [ ]:
# Cell 2: 配置 —— 只改 LOC 这一个值
from types import SimpleNamespace

DATA_ROOT = Path(r"D:\0_MAIN_BIKE_DATASETS_clean")

LOC = "25"          # ← 只改这里: "01" ~ "27", 变体写 "04-2" / "19-2"
MANUAL_COUNTS = None  # 人工计数基准, 例: (179, 62) 表示顺向179人/逆向62人; 没有则保持 None

LOC_ID  = f"loc_{LOC}"
IMG_DIR = DATA_ROOT / f"Loc_{LOC}" / "Bicyclist"
ROI_JSON = REPO_ROOT / "configs" / "locations_new" / f"{LOC_ID}.json"
OUTDIR   = REPO_ROOT / "outputs_new" / LOC_ID

args = SimpleNamespace(
    model      = "yolov8s.pt",   # 批跑标准配置; 更快但漏检多可换回 yolov8n.pt
    imgsz      = 1280,           # 高分辨率推理: 夜间/小目标召回关键
    conf       = 0.10,
    classes    = {1},
    nms_iou    = 0.70,
    assoc_gap  = 3,
    min_move_px= 25.0,          # 已标定: 静止目标抖动<13px, 真实骑行>50px
    cos_gate   = 0.5,
    # --- 时间/停驻修正 (机制性修正, 与manual无关) ---
    max_time_gap_s = 30.0,       # EXIF拍摄时间差超过30s强制断开关联(动态触发相机帧号连续≠时间连续); 0=关闭
    stationary_filter = True,    # 剔除停放自行车(同一位置反复出现+时间跨度长); 被剔除的存 stationary_objects.csv 供核查
    stationary_radius = 30.0,    # 停驻聚类半径(px)
    stationary_hits   = 6,       # 同一位置至少出现的独立帧数
    stationary_span_s = 300.0,   # 时间跨度阈值: 等红灯不会等5分钟
    stationary_span_frames = 50, # 无EXIF时的帧号跨度兜底
    save_crops = True,           # 导出rider裁剪图(朝向标注用) -> crops/
    save_viz   = True,           # 导出可视化标注图 -> viz/
    max_images = None,
)

print("LOC_ID:", LOC_ID)
print("IMG_DIR:", IMG_DIR, "| exists:", IMG_DIR.exists())
print("ROI_JSON:", ROI_JSON, "| exists:", ROI_JSON.exists())

In [ ]:
# Cell 3: 跑当前地点
summary = rrc.run_location(LOC_ID, IMG_DIR, ROI_JSON, OUTDIR, args)
summary

## 场景报告（论文级）

运行下面的 cell 生成本地点的完整报告：**QC 漏斗表**（每一步筛掉多少、为什么——答辩用）、**riders 明细表**、**person 复查队列**（按置信度排序，人工确认漏检骑行者）、四张出版级图（同时存为 300dpi PNG 到 `outputs_new/loc_XX/report/`，论文直接引用）。

In [ ]:
# Cell 4: 生成报告 — 表格
rep = rrc.generate_report(OUTDIR, ROI_JSON, LOC_ID, manual=MANUAL_COUNTS)

if "stats" in rep:
    display(rep["stats"])                  # 场景总览: 顺/逆人数, WW比例(置信区间), 占用, 可对照人工计数
display(rep["funnel"])                     # QC 漏斗: 从原始照片到 riders 的每一步
if "riders_table" in rep:
    display(rep["riders_table"])           # 计数明细
if "direction_table" in rep:
    display(rep["direction_table"])        # 方向判定表: 每行对应 direction_check/ 里一张人工验证图
if "person_review" in rep:
    display(rep["person_review"])          # 待人工确认的 person 候选 (crops_person/ 里有对应裁剪图)

In [ ]:
# Cell 5: 报告图 (自动渲染; PNG 已存至 report/ 文件夹)
import matplotlib.pyplot as plt
plt.show()
print("图表 PNG 已保存到:", OUTDIR / "report")

In [ ]:
# Cell 6: 生成三合一人工复查页 (朝向标注 + 候选确认 + precision 抽样)
out, nr, nc = rrc.generate_review_html(OUTDIR, LOC_ID)
print(f"复查页已生成: {out}")
print(f"  A区 riders 裁剪图: {nr} 个 (确认真实性+朝向)")
print(f"  B区 person 候选: {nc} 个 (确认漏检骑行者)")
print("用浏览器打开该 html, 逐个点按钮标注, 完成后点 Export CSV, 把 review_*.csv 发给 Claude")